# U-Net Segmentation Training on Google Colab

This notebook trains the repository's U-Net model with a `.tar.gz` dataset stored in Google Drive. The archive is copied to Colab, extracted to local storage, and checked before training starts. Checkpoints and plots are saved to Google Drive.

Before you begin, select **Runtime > Change runtime type > T4 GPU**. Push the latest training code to the configured Git branch, then run the cells from top to bottom.

## 1. Check the GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "A GPU is not available. Enable a GPU in the Colab runtime settings."
    )

print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive

The dataset archive and training outputs will be read from and written to Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Edit the experiment configuration

This is the main configuration cell. Edit the repository, archive, output, training, loss, and DataLoader settings here before running the remaining cells. The selected loss must stay unchanged when resuming an experiment.

In [ ]:
from pathlib import Path

# Repository settings
REPOSITORY_URL = "https://github.com/FillipusAditya/mask-guided-lung-nodule-xai.git"
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

# Dataset archive settings
DRIVE_ARCHIVE_PATH = Path(
    "/content/drive/MyDrive/mask-guided-lung-nodule-xai/"
    "segmentation_training_data.tar.gz"
)
LOCAL_ARCHIVE_PATH = Path("/content/segmentation_training_data.tar.gz")
EXTRACTION_ROOT = Path("/content/segmentation_training_data")
DATASET_ROOT = EXTRACTION_ROOT / "000_dataset_v2/_segmentation_dataset"
COPY_ARCHIVE_TO_LOCAL = True
FORCE_EXTRACT = False

# Persistent output settings
DRIVE_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/mask-guided-lung-nodule-xai/experiment_results"
)

# Dataset settings
METADATA_FILENAME = "001_holdout_split_lidc_lndb.csv"
IMAGE_PATH_COLUMN = "ct_parenchyma_path"
INPUT_HEIGHT = 512
INPUT_WIDTH = 512
TILE_GRID_SIZE = 4

# Training settings
LEARNING_RATE = 1e-3
BATCH_SIZE = 8  # Increase gradually after checking A100 memory usage.
TOTAL_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 5
SEED = 42
PREDICTION_THRESHOLD = 0.5

# Accelerator settings
PARALLEL_TILE_PROCESSING = True
TILE_CHUNK_SIZE = None  # Use 64 or 32 to limit the size of each model call.
ACTIVATION_CHECKPOINTING = False
VALIDATION_ON_GPU = True
NON_BLOCKING_TRANSFER = True

# Loss settings: DiceLoss, BCEDiceLoss, or IoULoss
LOSS_NAME = "BCEDiceLoss"
LOSS_SMOOTH = 1e-6

# Optimizer settings
WEIGHT_DECAY = 1e-4

# DataLoader and mixed-precision settings
NUM_WORKERS = 6
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 2
TRAIN_SHUFFLE = True
AMP_ENABLED = True

# Use None for a new experiment. For resume, use the last checkpoint path.
RESUME_CHECKPOINT_PATH = None
# Example:
# RESUME_CHECKPOINT_PATH = Path(
#     "/content/drive/MyDrive/mask-guided-lung-nodule-xai/"
#     "experiment_results/<run-id>/segmentation/unet/last_checkpoint.pth"
# )

## 4. Clone or update the repository

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    print("Updating the existing repository...")
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git", "-C", str(PROJECT_ROOT), "pull", "--ff-only",
            "origin", REPOSITORY_BRANCH,
        ],
        check=True,
    )
else:
    print("Cloning the repository...")
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

print(f"Repository ready: {PROJECT_ROOT}")

## 5. Install the required packages

Colab's existing PyTorch installation is kept because it matches the active CUDA runtime.

In [ ]:
import sys

packages = [
    "albumentations>=2.0,<3.0",
    "opencv-python-headless",
    "pandas",
    "matplotlib",
    "tqdm",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

print("Dependencies are ready.")

## 6. Copy and extract the `.tar.gz` dataset

The copy and extraction operations display `tqdm` progress bars. The expected archive contains a top-level `000_dataset_v2` directory. Set `FORCE_EXTRACT=True` only when you need to replace an existing local extraction.

In [ ]:
import shutil
import tarfile

from tqdm.auto import tqdm


def copy_file_with_progress(source, destination, chunk_size=8 * 1024 * 1024):
    """Copy one file and show byte-level progress."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_destination = destination.with_suffix(destination.suffix + ".part")
    total_bytes = source.stat().st_size

    with source.open("rb") as input_file:
        with temporary_destination.open("wb") as output_file:
            with tqdm(
                total=total_bytes,
                desc="Copying archive",
                unit="B",
                unit_scale=True,
            ) as progress_bar:
                while True:
                    chunk = input_file.read(chunk_size)
                    if not chunk:
                        break
                    output_file.write(chunk)
                    progress_bar.update(len(chunk))

    temporary_destination.replace(destination)


if not DRIVE_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DRIVE_ARCHIVE_PATH}")

if COPY_ARCHIVE_TO_LOCAL:
    archive_path = LOCAL_ARCHIVE_PATH
    source_size = DRIVE_ARCHIVE_PATH.stat().st_size
    local_copy_is_current = (
        archive_path.is_file() and archive_path.stat().st_size == source_size
    )

    if not local_copy_is_current:
        copy_file_with_progress(DRIVE_ARCHIVE_PATH, archive_path)
    else:
        print(f"Reusing the local archive: {archive_path}")
else:
    archive_path = DRIVE_ARCHIVE_PATH

extraction_marker = EXTRACTION_ROOT / ".extraction_complete"

if FORCE_EXTRACT and EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

if not extraction_marker.is_file():
    EXTRACTION_ROOT.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive_path, mode="r:gz") as archive:
        members = archive.getmembers()
        for member in tqdm(members, desc="Extracting dataset", unit="file"):
            archive.extract(member, path=EXTRACTION_ROOT, filter="data")

    extraction_marker.touch()
else:
    print(f"Reusing the extracted dataset: {EXTRACTION_ROOT}")

print(f"Dataset root: {DATASET_ROOT}")

## 7. Validate the extracted dataset

This cell checks the required directories and verifies every image and mask path referenced by the holdout metadata.

In [ ]:
import csv

metadata_path = DATASET_ROOT / METADATA_FILENAME
required_directories = [
    EXTRACTION_ROOT / "000_dataset_v2/_lidc/007_segmentation_dataset_npy/ct_parenchyma",
    EXTRACTION_ROOT / "000_dataset_v2/_lidc/007_segmentation_dataset_npy/mask",
    EXTRACTION_ROOT / "000_dataset_v2/_lndb/007_segmentation_dataset_npy/ct_parenchyma",
    EXTRACTION_ROOT / "000_dataset_v2/_lndb/007_segmentation_dataset_npy/mask",
]

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata file not found: {metadata_path}")

for directory in required_directories:
    if not directory.is_dir():
        raise FileNotFoundError(f"Required directory not found: {directory}")

with metadata_path.open("r", encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

missing_files = []
for row in tqdm(rows, desc="Checking dataset paths", unit="sample"):
    image_path = DATASET_ROOT / row[IMAGE_PATH_COLUMN]
    mask_path = DATASET_ROOT / row["mask_path"]

    if not image_path.is_file():
        missing_files.append(image_path)
    if not mask_path.is_file():
        missing_files.append(mask_path)

if missing_files:
    examples = "\n".join(str(path) for path in missing_files[:5])
    raise FileNotFoundError(
        f"The archive is missing {len(missing_files)} referenced files.\n{examples}"
    )

print(f"Metadata rows: {len(rows):,}")
print("All referenced images and masks are available.")

## 8. Write the training configuration

This cell creates the complete JSON configuration used by `train.py`. It only modifies the temporary repository clone in the current Colab runtime.

In [ ]:
import json

supported_losses = {"DiceLoss", "BCEDiceLoss", "IoULoss"}
if LOSS_NAME not in supported_losses:
    raise ValueError(
        f"LOSS_NAME must be one of {sorted(supported_losses)}, not {LOSS_NAME!r}."
    )

if RESUME_CHECKPOINT_PATH is not None:
    RESUME_CHECKPOINT_PATH = Path(RESUME_CHECKPOINT_PATH)
    if not RESUME_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(
            f"Resume checkpoint not found: {RESUME_CHECKPOINT_PATH}"
        )
    if not (RESUME_CHECKPOINT_PATH.parent / "training_log.csv").is_file():
        raise FileNotFoundError(
            f"Training log not found beside: {RESUME_CHECKPOINT_PATH}"
        )

config = {
    "output": {"root_directory": str(DRIVE_OUTPUT_ROOT)},
    "data": {
        "dataset_root": str(DATASET_ROOT),
        "split_method": "holdout_split",
        "metadata_filename": METADATA_FILENAME,
        "fold": None,
        "image_path_column": IMAGE_PATH_COLUMN,
        "input_height": INPUT_HEIGHT,
        "input_width": INPUT_WIDTH,
        "tile_grid_size": TILE_GRID_SIZE,
    },
    "training": {
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "total_epochs": TOTAL_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "seed": SEED,
        "prediction_threshold": PREDICTION_THRESHOLD,
        "parallel_tile_processing": PARALLEL_TILE_PROCESSING,
        "tile_chunk_size": TILE_CHUNK_SIZE,
        "activation_checkpointing": ACTIVATION_CHECKPOINTING,
        "validation_on_gpu": VALIDATION_ON_GPU,
        "non_blocking_transfer": NON_BLOCKING_TRANSFER,
    },
    "loss": {"name": LOSS_NAME, "smooth": LOSS_SMOOTH},
    "optimizer": {"weight_decay": WEIGHT_DECAY},
    "dataloader": {
        "num_workers": NUM_WORKERS,
        "persistent_workers": PERSISTENT_WORKERS,
        "prefetch_factor": PREFETCH_FACTOR,
        "train_shuffle": TRAIN_SHUFFLE,
    },
    "amp": {"training_enabled": AMP_ENABLED},
    "checkpoint": {
        "resume_checkpoint_path": (
            str(RESUME_CHECKPOINT_PATH)
            if RESUME_CHECKPOINT_PATH is not None
            else None
        )
    },
}

config_path = PROJECT_ROOT / "002_segmentation/configs/unet_holdout.json"
with config_path.open("w", encoding="utf-8") as file:
    json.dump(config, file, indent=4)

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(json.dumps(config, indent=4))

## 9. Run a dataset preflight check

The preflight uses the same training transform as `train.py`, loads one training sample, and confirms that its transformed mask remains binary.

In [ ]:
import runpy

segmentation_root = (PROJECT_ROOT / "002_segmentation").resolve()
if str(segmentation_root) not in sys.path:
    sys.path.insert(0, str(segmentation_root))

training_script = segmentation_root / "unet_holdout/train.py"
training_module = runpy.run_path(
    str(training_script),
    run_name="unet_training_preflight",
)

from unet_utils.dataset import LungDataset

dataset_arguments = {
    "root_dir": DATASET_ROOT,
    "split_method": "holdout_split",
    "metadata_filename": METADATA_FILENAME,
    "fold": None,
    "image_path_column": IMAGE_PATH_COLUMN,
    "tile_grid_size": TILE_GRID_SIZE,
}

train_dataset = LungDataset(
    split="train",
    transform=training_module["build_train_transform"](),
    **dataset_arguments,
)
val_dataset = LungDataset(
    split="val",
    transform=training_module["build_val_transform"](),
    **dataset_arguments,
)

if len(train_dataset) == 0 or len(val_dataset) == 0:
    raise RuntimeError("The training or validation split is empty.")

image_tiles, mask_tiles = train_dataset[0]
mask_values = torch.unique(mask_tiles).tolist()
if not set(mask_values).issubset({0.0, 1.0}):
    raise ValueError(f"The transformed mask is not binary: {mask_values}")

print(f"Training samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Image tile shape: {tuple(image_tiles.shape)}")
print(f"Mask tile shape: {tuple(mask_tiles.shape)}")
print(f"Mask values: {mask_values}")

## 10. Start training

Training runs inside the notebook kernel, so the `tqdm` training and validation progress bars are displayed directly below this cell. A checkpoint is saved after every completed epoch.

In [ ]:
print(f"Starting training with {LOSS_NAME}...", flush=True)
runpy.run_path(str(training_script), run_name="__main__")

## 11. Inspect the latest result

This cell shows the last log rows, generated plots, and the checkpoint path to use for a future resume.

In [ ]:
import pandas as pd
from IPython.display import Image, display

if RESUME_CHECKPOINT_PATH is not None:
    latest_output_dir = RESUME_CHECKPOINT_PATH.parent
else:
    checkpoints = list(
        DRIVE_OUTPUT_ROOT.glob("*/segmentation/unet/last_checkpoint.pth")
    )
    if not checkpoints:
        raise FileNotFoundError("No completed training checkpoint was found.")

    latest_checkpoint = max(checkpoints, key=lambda path: path.stat().st_mtime)
    latest_output_dir = latest_checkpoint.parent

training_log_path = latest_output_dir / "training_log.csv"
last_checkpoint_path = latest_output_dir / "last_checkpoint.pth"

print(f"Output directory: {latest_output_dir}")
display(pd.read_csv(training_log_path).tail())

plot_names = [
    "loss_curve.png",
    "dice_curve.png",
    "iou_curve.png",
    "metrics_curve.png",
]
for plot_name in plot_names:
    plot_path = latest_output_dir / "visualizations" / plot_name
    if plot_path.is_file():
        display(Image(filename=str(plot_path), width=650))

print("Use this path to resume in another Colab session:")
print(last_checkpoint_path)

## Resuming in a new Colab session

1. Run the GPU, Google Drive, configuration, repository, dependency, and extraction cells again.
2. Set `RESUME_CHECKPOINT_PATH` in the configuration cell to the printed `last_checkpoint.pth` path.
3. Keep `LOSS_NAME` and `LOSS_SMOOTH` identical to the saved experiment.
4. Set `TOTAL_EPOCHS` to the final target epoch, not the number of additional epochs.
5. Run the configuration-writing, preflight, and training cells. If training is already complete, the script will regenerate its visualizations.